# Chapter 6 (জীবে পরিবহন) — triple extraction

Vertical slice. One chapter, extracted into subject–relation–object triples, so we can
look at real output before committing to a schema or scaling to 14 chapters.

**Deliberately not fixed in advance:** the relation vocabulary. The model proposes
relations freely and we read off what it actually produces — designing the schema first
would mean guessing what Bangla biology prose contains.

**Setup on Kaggle**
- Accelerator: GPU T4 x2 (or P100)
- Add data: the output of `02_biology_normalize` (for `biology_9_10_clean.parquet`)
- Add model: `qwen-lm/qwen2.5` → `transformers` → `7b-instruct`

In [ ]:
import json, re, glob, time
from pathlib import Path
import pandas as pd

CHAPTER = 6

hits = glob.glob("/kaggle/input/**/biology_9_10_clean.parquet", recursive=True) \
     + glob.glob("/kaggle/working/biology_9_10_clean.parquet") \
     + glob.glob("biology_9_10_clean.parquet")
if not hits:
    raise SystemExit(
        "biology_9_10_clean.parquet not found. Run 02_biology_normalize, Save Version, "
        "then add that notebook's output as a data source here."
    )

df = pd.read_parquet(hits[0])
ch = df[df.chapter_no == CHAPTER].reset_index(drop=True)
print(f"{hits[0]}\n{len(ch)} chunks — {ch.chapter_title.iloc[0]}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_dirs = glob.glob("/kaggle/input/qwen*/**/7b-instruct/**/config.json", recursive=True)
if not model_dirs:
    model_dirs = glob.glob("/kaggle/input/**/config.json", recursive=True)
if not model_dirs:
    raise SystemExit("No model found — add qwen-lm/qwen2.5 (transformers / 7b-instruct).")
MODEL_PATH = str(Path(model_dirs[0]).parent)
print("model:", MODEL_PATH)

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    ),
    device_map="auto",
)
model.eval()
print("loaded")

## Prompt

Entities stay in Bangla verbatim — the KG has to match against Bangla tutor answers later,
so translating here would just add a lossy hop. Relations are also requested in Bangla for
the same reason, and left unconstrained so we can see the natural vocabulary.

In [ ]:
SYSTEM = "তুমি একজন পাঠ্যবই বিশ্লেষক। তুমি শুধুমাত্র বৈধ JSON উত্তর দাও।"

PROMPT = """নিচে বাংলাদেশের নবম-দশম শ্রেণির জীববিজ্ঞান পাঠ্যবইয়ের একটি অংশ দেওয়া হলো।

এই অংশে যেসব তথ্য স্পষ্টভাবে বলা হয়েছে, সেগুলো (subject, relation, object) আকারে বের করো।

নিয়ম:
- শুধু পাঠ্যাংশে যা সরাসরি বলা আছে তাই নাও। নিজে থেকে কিছু যোগ করবে না।
- subject, relation, object সবই বাংলায় লিখবে, বইয়ের শব্দ হুবহু রাখবে।
- relation ছোট ক্রিয়াপদ বা সম্পর্কবাচক শব্দ হবে, যেমন: অংশ, প্রকার, কাজ, সংজ্ঞা, কারণ, অবস্থান।
- সংখ্যা বা পরিমাপ থাকলে object-এ একক সহ রাখবে।
- কোনো তথ্য না থাকলে খালি তালিকা দেবে।

শুধু এই ফরম্যাটে JSON দাও, অন্য কোনো লেখা নয়:
{{"triples": [{{"subject": "...", "relation": "...", "object": "..."}}]}}

পাঠ্যাংশ:
\"\"\"{chunk}\"\"\""""


def build(chunk):
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": PROMPT.format(chunk=chunk)}],
        tokenize=False, add_generation_prompt=True,
    )

print(build(ch.text.iloc[0])[:600])

In [ ]:
@torch.inference_mode()
def generate(prompts, max_new_tokens=512, batch_size=4):
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    out = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(batch)):
            out.append(tok.decode(gen[j][enc.input_ids.shape[1]:],
                                  skip_special_tokens=True))
        print(f"  {min(i+batch_size, len(prompts))}/{len(prompts)}", end="\r")
    return out


# Start with a handful. Only scale to the full chapter once the output looks sane.
SAMPLE = 8
sample = ch.head(SAMPLE)

t0 = time.time()
raw = generate([build(t) for t in sample.text])
print(f"\n{len(raw)} chunks in {time.time()-t0:.0f}s")
print(raw[0][:700])

## Parsing

Models wrap JSON in prose or fences even when told not to, so the parser recovers the
outermost object rather than trusting the response wholesale, and records failures instead
of dropping them silently — a chunk that never parses is a data point about the prompt.

In [ ]:
def extract_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    start = text.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i, c in enumerate(text[start:], start):
        if esc:
            esc = False
        elif c == "\\":
            esc = True
        elif c == '"':
            in_str = not in_str
        elif not in_str:
            depth += (c == "{") - (c == "}")
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None


rows, failed = [], []
for chunk_id, text in zip(sample.chunk_id, raw):
    obj = extract_json(text)
    if not obj or "triples" not in obj:
        failed.append((chunk_id, text[:200]))
        continue
    for t in obj["triples"]:
        if isinstance(t, dict) and {"subject", "relation", "object"} <= t.keys():
            rows.append({"chunk_id": chunk_id, **{k: str(t[k]).strip()
                                                  for k in ("subject", "relation", "object")}})

triples = pd.DataFrame(rows)
print(f"{len(triples)} triples from {len(sample)} chunks; {len(failed)} unparseable")
for cid, snippet in failed:
    print(f"  FAILED {cid}: {snippet!r}")
triples

## What relations did it actually invent?

This is the output that decides the schema. A long tail of near-duplicate relations means
the vocabulary needs constraining on the next pass.

In [ ]:
if len(triples):
    print(triples.relation.value_counts().to_string())
    print(f"\n{triples.subject.nunique()} distinct subjects, "
          f"{triples.object.nunique()} distinct objects, "
          f"{triples.relation.nunique()} distinct relations")

In [ ]:
triples.to_csv("/kaggle/working/ch6_triples_sample.csv", index=False)
pd.DataFrame({"chunk_id": sample.chunk_id, "raw": raw}).to_json(
    "/kaggle/working/ch6_raw_output.jsonl", orient="records", lines=True, force_ascii=False)
print("saved")